In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from shadow_peft import AutoModelForCausalLMWithHiddenProjection, ShadowForCausalLM

shadow_peft_id = "/home/lxm/workspace/Shadow/outputs/gsm8k_suite_pretrained_shadow_8B-2-5/gsm8k_gsm8k_main/shadow_peft/"  # '/home/lxm/workspace/Shadow/outputs/squad_v2_suite_shadow_4B/squad_v2_squad_v2/shadow_peft'
shadow_model_id = 'erin99/Qwen3-0.6B-H8B'
backbone_model_id = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(backbone_model_id)
base_model = AutoModelForCausalLM.from_pretrained(backbone_model_id, torch_dtype=torch.bfloat16).to('cuda:6')

shadow_model = AutoModelForCausalLMWithHiddenProjection.from_pretrained(shadow_model_id).to(base_model.device, dtype=base_model.dtype)
model = ShadowForCausalLM.from_pretrained(
    base_model,
    shadow_peft_id,
    is_trainable=False,
    shadow_model=shadow_model
).to(base_model.device, dtype=base_model.dtype)

model = model.eval()

/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 104.74it/s]


'\nshadow_model = AutoModelForCausalLMWithHiddenProjection.from_pretrained(shadow_model_id).to(base_model.device, dtype=base_model.dtype)\nmodel = ShadowForCausalLM.from_pretrained(\n    base_model,\n    shadow_peft_id,\n    is_trainable=False,\n    shadow_model=shadow_model\n).to(base_model.device, dtype=base_model.dtype)\n\nmodel = model.eval()\n'

In [24]:
import torch

# Example GSM8K problem
# question = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
question = "1+1=?"
question = "Ariel was playing basketball. 1 of her shots went in the hoop. 2 of her shots did not go in the hoop. How many shots were there in total?"
question = 'our team scored a total of 123 points. 67 points were scored in the first half. How many were scored in the second half?'
question = "小王的妈妈有三个儿子，大儿子叫大明，二儿子叫二明，三儿子叫什么？"
# question = "9.3 and 9.200 which is bigger?"
question = "blood relationship identify: Grand Mother's Brother"
question = '有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？'
question = "我想洗车，洗车店距离我50m，你推荐我是开车去还是走路去？（不要思考直接回答）"

user_content = f"Question: {question}"
user_message = {"role": "user", "content": user_content}
prompt = tokenizer.apply_chat_template(
    [user_message],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

# Tokenize and generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

# Decode and print the result
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


user
Question: 我想洗车，洗车店距离我50m，你推荐我是开车去还是走路去？（不要思考直接回答）
assistant
<think>
开车去更快，因为50m的距离走路需要5分钟，开车只需要1分钟。
#### 1
